<a href="https://colab.research.google.com/github/mindioanni/LandLevelTools/blob/main/Adjust_Height_Network_V5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === Search Drive Tree for XLSX & Configure Input ===
# Name: search_xlsx_and_configure
# Description:
# - Mount Google Drive
# - Recursively search for .xlsx under a root folder (whole tree)
# - Display matches as relative paths (distinguish duplicate filenames)
# - Optional keyword filter (e.g., ASTY)
# - Select file -> load sheet -> choose HI column -> record observation date knowledge

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

def _normalize_colname(c: str) -> str:
    return " ".join(str(c).strip().split())

def find_xlsx_files(root_dir: str) -> list[str]:
    """Return sorted list of full paths to .xlsx files under root_dir (recursive)."""
    out = []
    for dirpath, _, filenames in os.walk(root_dir):
        for fn in filenames:
            if fn.lower().endswith(".xlsx"):
                out.append(os.path.join(dirpath, fn))
    out.sort(key=lambda p: p.lower())
    return out

# 1) Root for recursive search
root_dir = input(
    "\nEnter root folder to search recursively\n"
    "If uncertain provide: /content/drive/MyDrive\n> "
).strip()

if not os.path.isdir(root_dir):
    raise FileNotFoundError(f"Folder not found: {root_dir}")

print("\nSearching... (this may take time if root is large)")
all_xlsx = find_xlsx_files(root_dir)
print(f"Found {len(all_xlsx)} .xlsx files under: {root_dir}")

# 2) Optional keyword filtering
kw = input("\nOptional filter keyword (e.g., ASTY). Press Enter to skip: ").strip()
if kw:
    kw_low = kw.lower()
    filtered = [p for p in all_xlsx if kw_low in os.path.relpath(p, root_dir).lower()]
    print(f"Filtered to {len(filtered)} files containing '{kw}'.")
else:
    filtered = all_xlsx

if len(filtered) == 0:
    raise RuntimeError("No .xlsx files found after filtering.")

# 3) Display a selectable list (relative paths for disambiguation)
max_show = 200
print("\nAvailable XLSX files (showing up to first 200):")
for i, p in enumerate(filtered[:max_show], start=1):
    rel = os.path.relpath(p, root_dir)
    print(f"  [{i:03d}] {rel}")

if len(filtered) > max_show:
    print(f"\nNote: {len(filtered)} files total. Refine with a keyword to reduce the list.")

sel = input("\nSelect file number from the list above: ").strip()
idx = int(sel) - 1
if idx < 0 or idx >= len(filtered[:max_show]):
    raise ValueError("Selection out of range for the displayed list. Refine filter or increase max_show in code.")

xlsx_path = filtered[idx]
print("\nSelected XLSX path:", xlsx_path)

# 4) Load workbook and select sheet
xls = pd.ExcelFile(xlsx_path)
sheets = xls.sheet_names

print("\nWorksheets found:")
for i, s in enumerate(sheets, start=1):
    print(f"  [{i}] {s}")

sheet_idx = input("\nSelect worksheet number (press Enter for 1): ").strip()
sheet_name = sheets[0] if sheet_idx == "" else sheets[int(sheet_idx) - 1]
print(f"\nSelected worksheet: {sheet_name}")

df_raw = pd.read_excel(xlsx_path, sheet_name=sheet_name)
df_raw.columns = [_normalize_colname(c) for c in df_raw.columns]

print(f"\nLoaded rows: {len(df_raw)}")
print(f"Loaded columns: {len(df_raw.columns)}")

# 5) Choose instrument height column (HI)
candidates_hi = []
for c in df_raw.columns:
    cu = c.upper()
    if ("INSTRUMENT HEIGHT" in cu) or ("CORRECTED" in cu and "HEIGHT" in cu) or (cu in ["HI", "H.I.", "H_I"]):
        candidates_hi.append(c)

print("\nInstrument height column candidates:")
if len(candidates_hi) == 0:
    print("  (none detected)")
    hi_col = input("Type the exact column name to use for instrument height (HI): ").strip()
    if hi_col not in df_raw.columns:
        raise ValueError(f"Column not found: {hi_col}")
elif len(candidates_hi) == 1:
    hi_col = candidates_hi[0]
    print(f"  Using: {hi_col}")
else:
    for i, c in enumerate(candidates_hi, start=1):
        print(f"  [{i}] {c}")
    sel_hi = input("Select which column to use for instrument height (HI): ").strip()
    hi_col = candidates_hi[int(sel_hi) - 1]

# 6) Ask if user knows observation date
knows_date = input("\nDo you know the observation date for this file? (y/n): ").strip().lower()
if knows_date not in ["y", "n"]:
    raise ValueError("Please answer with 'y' or 'n'.")

obs_date_str = None
if knows_date == "y":
    obs_date_str = input("Enter date as YYYY-MM-DD (e.g., 2025-10-27): ").strip()

# 7) Save configuration for next cells
config = {
    "xlsx_path": xlsx_path,
    "sheet_name": sheet_name,
    "hi_col": hi_col,
    "knows_date": (knows_date == "y"),
    "obs_date_str": obs_date_str,
    "search_root": root_dir,
    "filter_keyword": kw if kw else None,
}

print("\nConfiguration saved in variable: config")
print(config)
print("\nDataFrame saved in variable: df_raw")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Enter root folder to search recursively
If uncertain provide: /content/drive/MyDrive
> /content/drive/MyDrive

Searching... (this may take time if root is large)
Found 28 .xlsx files under: /content/drive/MyDrive

Optional filter keyword (e.g., ASTY). Press Enter to skip: ASTY
Filtered to 5 files containing 'ASTY'.

Available XLSX files (showing up to first 200):
  [001] ΕΡΕΥΝΑ/Γεωδυναμικό Ινστιτούτο/EPOS TCS TSU/ΔΕΔΟΜΕΝΑ ΠΕΔΙΟΥ/ASTY/ASTYPALAIA DATA.xlsx
  [002] ΕΡΕΥΝΑ/Γεωδυναμικό Ινστιτούτο/EPOS TCS TSU/ΔΕΔΟΜΕΝΑ ΠΕΔΙΟΥ/ASTY/EpilisimeTaximetria/Project1891.xlsx
  [003] ΕΡΕΥΝΑ/Γεωδυναμικό Ινστιτούτο/EPOS TCS TSU/ΔΕΔΟΜΕΝΑ ΠΕΔΙΟΥ/ASTY/PYTHON PROGRAM/Reciprocal.xlsx
  [004] ΕΡΕΥΝΑ/Γεωδυναμικό Ινστιτούτο/EPOS TCS TSU/ΔΕΔΟΜΕΝΑ ΠΕΔΙΟΥ/ASTY/PYTHON PROGRAM/Reciprocal_plus_targets_FINAL.xlsx
  [005] ΕΡΕΥΝΑ/Γεωδυναμικό Ινστιτούτο/EPOS TCS TSU/ΔΕΔΟΜΕΝΑ ΠΕΔΙΟΥ/

In [ ]:
# === Normalize Observations Table ===
# Name: normalize_observations_table
# Description:
# - Build a standardized observation DataFrame `obs` from `df_raw` using `config`
# - Parse numeric fields and Face (I/II)
# - Build datetime (from Date/Time columns if present, otherwise from user-provided obs_date_str)
# - Flag invalid rows (missing critical fields) and report counts

import numpy as np
import pandas as pd

# ---------- Helpers ----------
def parse_float(x):
    """Parse floats from numbers or strings like '223.4311 gon' or '-'."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip()
    if s in ["", "-", "—", "–", "nan", "NaN", "None"]:
        return np.nan
    # keep first numeric token
    s = s.replace(",", ".")
    token = s.split()[0]
    try:
        return float(token)
    except Exception:
        # fallback: extract numeric pattern
        import re
        m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
        return float(m.group(0)) if m else np.nan

def parse_face(x):
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    if s in ["I", "1", "F1", "FACE I", "FACEI", "FI"]:
        return "I"
    if s in ["II", "2", "F2", "FACE II", "FACEII", "FII"]:
        return "II"
    return None

def find_col(df, candidates):
    """Return first matching column in df for a list of candidate names."""
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None

# ---------- Required column mapping (best-effort) ----------
col_from = find_col(df_raw, ["From Station", "From", "Station", "Station Name", "Setup Station"])
col_to   = find_col(df_raw, ["To", "Target", "To Station", "Point", "Sight To"])
col_face = find_col(df_raw, ["Face", "FACE"])
col_v    = find_col(df_raw, ["V [gon]", "V(gon)", "V", "Zenith [gon]", "Zenith", "Z [gon]"])
col_sd   = find_col(df_raw, ["Slope Dist. [m]", "Slope Dist [m]", "Slope Distance [m]", "Slope [m]", "SD [m]"])
col_ht   = find_col(df_raw, ["Target Height [m]", "Target Height", "HT [m]", "Prism Height [m]", "Reflector Height [m]"])

# Date/time (optional)
col_date = find_col(df_raw, ["Date", "DATE"])
col_time = find_col(df_raw, ["Time", "TIME", "Start Time"])

missing = []
for name, col in [("From Station", col_from), ("To", col_to), ("Face", col_face),
                  ("V [gon]", col_v), ("Slope Dist. [m]", col_sd), ("Target Height [m]", col_ht)]:
    if col is None:
        missing.append(name)

if missing:
    raise ValueError(f"Missing required columns (not found by auto-detection): {missing}\n"
                     f"Available columns: {list(df_raw.columns)}")

hi_col = config["hi_col"]
if hi_col not in df_raw.columns:
    raise ValueError(f"HI column from config not found in df_raw: {hi_col}")

# ---------- Build obs ----------
obs = pd.DataFrame()
obs["from"] = df_raw[col_from].astype(str).str.strip()
obs["to"]   = df_raw[col_to].astype(str).str.strip()
obs["face"] = df_raw[col_face].apply(parse_face)

obs["V_gon"] = df_raw[col_v].apply(parse_float)
obs["slope_m"] = df_raw[col_sd].apply(parse_float)
obs["HI_m"] = df_raw[hi_col].apply(parse_float)
obs["HT_m"] = df_raw[col_ht].apply(parse_float)

# ---------- Datetime handling ----------
# If Date & Time exist -> combine; else use user-provided obs_date_str; else NaT.
if col_date is not None and col_time is not None:
    # robust combine; keep as string then to_datetime
    dt_str = df_raw[col_date].astype(str).str.strip() + " " + df_raw[col_time].astype(str).str.strip()
    obs["datetime"] = pd.to_datetime(dt_str, errors="coerce", dayfirst=False)
elif config.get("knows_date", False) and config.get("obs_date_str"):
    # store date only; time unknown
    obs["datetime"] = pd.to_datetime(config["obs_date_str"], errors="coerce")
else:
    obs["datetime"] = pd.NaT

# ---------- Validity flags ----------
# Critical fields for next steps
crit = ["from", "to", "face", "V_gon", "slope_m", "HI_m", "HT_m"]
valid = np.ones(len(obs), dtype=bool)

# Missing from/to
valid &= obs["from"].notna() & (obs["from"].str.len() > 0)
valid &= obs["to"].notna() & (obs["to"].str.len() > 0)

# Face must be I or II
valid &= obs["face"].isin(["I", "II"])

# Numeric fields not NaN
for c in ["V_gon", "slope_m", "HI_m", "HT_m"]:
    valid &= np.isfinite(obs[c].values)

obs["is_valid"] = valid

# ---------- Report ----------
n_total = len(obs)
n_valid = int(obs["is_valid"].sum())
n_invalid = n_total - n_valid

print(f"Total rows:   {n_total}")
print(f"Valid rows:   {n_valid}")
print(f"Invalid rows: {n_invalid}")

if n_invalid > 0:
    # simple diagnostics
    reasons = {
        "missing/empty from": ~(obs["from"].notna() & (obs["from"].str.len() > 0)),
        "missing/empty to":   ~(obs["to"].notna() & (obs["to"].str.len() > 0)),
        "invalid face":       ~obs["face"].isin(["I", "II"]),
        "V_gon NaN":          ~np.isfinite(obs["V_gon"].values),
        "slope_m NaN":        ~np.isfinite(obs["slope_m"].values),
        "HI_m NaN":           ~np.isfinite(obs["HI_m"].values),
        "HT_m NaN":           ~np.isfinite(obs["HT_m"].values),
    }
    print("\nInvalid-row reasons (counts):")
    for k, m in reasons.items():
        print(f" - {k}: {int(m.sum())}")

# Keep only valid observations for downstream processing
obs = obs.loc[obs["is_valid"]].copy()

# Standard sorting for reproducibility
if obs["datetime"].notna().any():
    obs = obs.sort_values(["datetime", "from", "to", "face"]).reset_index(drop=True)
else:
    obs = obs.sort_values(["from", "to", "face"]).reset_index(drop=True)

print("\nSaved standardized observations in DataFrame: obs")
print("Columns:", list(obs.columns))
obs.head(10)


Total rows:   278
Valid rows:   278
Invalid rows: 0

Saved standardized observations in DataFrame: obs
Columns: ['from', 'to', 'face', 'V_gon', 'slope_m', 'HI_m', 'HT_m', 'datetime', 'is_valid']


,from,to,face,V_gon,slope_m,HI_m,HT_m,datetime,is_valid
0,R1,R2,I,108.0436,21.6415,1.51896,0.1,2025-01-19,True
1,R1,R2,I,108.0442,21.6415,1.51896,0.1,2025-01-19,True
2,R1,R2,I,108.0444,21.6408,1.51896,0.1,2025-01-19,True
3,R1,R2,I,108.0454,21.6420,1.51896,0.1,2025-01-19,True
4,R1,R2,I,108.0438,21.6422,1.51896,0.1,2025-01-19,True
5,R1,R2,I,108.0437,21.6422,1.51896,0.1,2025-01-19,True
6,R1,R2,I,108.0440,21.6420,1.51896,0.1,2025-01-19,True
7,R1,R2,I,108.0447,21.6421,1.51896,0.1,2025-01-19,True
8,R1,R2,I,108.0442,21.6421,1.51896,0.1,2025-01-19,True
9,R1,R2,I,108.0440,21.6422,1.51896,0.1,2025-01-19,True


In [ ]:
# === Compute Vertical Component & Height Difference ===
# Name: compute_vdist_and_dh
# Description:
# - Compute V_distC_m = d * cos(Z) where Z is zenith angle in radians (V_gon assumed zenith)
# - Compute dH_C_m = HI - HT + V_distC_m
# - If df_raw contains "V Dist. [m]" and/or "ΔHeight [m]" (or similar), compare and report stats

import numpy as np
import pandas as pd

# --- Assumption check (explicit): V_gon is zenith angle Z in gon ---
# Z [gon] -> radians: Z_rad = Z_gon * (pi/200)
GON_TO_RAD = np.pi / 200.0

# Compute vertical component (+ up if Z < 100 gon, - down if Z > 100 gon)
obs["Z_rad"] = obs["V_gon"].values * GON_TO_RAD
obs["V_distC_m"] = obs["slope_m"].values * np.cos(obs["Z_rad"].values)

# Height difference from S1 (from) to S2 (to): H_to - H_from
obs["dH_C_m"] = obs["HI_m"].values - obs["HT_m"].values + obs["V_distC_m"].values

# --- Optional comparison to raw-file columns (if exist) ---
def find_col(df, candidates):
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None

def parse_float(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip()
    if s in ["", "-", "—", "–", "nan", "NaN", "None"]:
        return np.nan
    s = s.replace(",", ".")
    token = s.split()[0]
    try:
        return float(token)
    except Exception:
        import re
        m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
        return float(m.group(0)) if m else np.nan

col_vdist_raw = find_col(df_raw, ["V Dist. [m]", "V Dist [m]", "V Dist", "V Distance [m]", "Vertical Dist. [m]"])
col_dh_raw    = find_col(df_raw, ["ΔHeight [m]", "dHeight [m]", "Delta Height [m]", "DeltaHeight [m]", "Dh [m]"])

# Build raw vectors aligned by row index (obs was built from df_raw row order and then sorted;
# so we compare only if obs is not re-ordered by datetime with identical date only. We will compare via a safe merge key.)
# Since your datetime is constant (date-only), sorting may have changed order. We'll compare by (from,to,face,V_gon,slope,HI,HT) key.

compare_report = {}

if col_vdist_raw is not None or col_dh_raw is not None:
    # create a key in both frames
    tmp_raw = df_raw.copy()
    tmp_raw["_from"] = tmp_raw[find_col(df_raw, ["From Station"])].astype(str).str.strip()
    tmp_raw["_to"]   = tmp_raw[find_col(df_raw, ["To"])].astype(str).str.strip()
    tmp_raw["_face"] = tmp_raw[find_col(df_raw, ["Face"])].astype(str).str.strip().str.upper()
    tmp_raw["_V_gon"] = tmp_raw[find_col(df_raw, ["V [gon]"])].apply(parse_float)
    tmp_raw["_slope_m"] = tmp_raw[find_col(df_raw, ["Slope Dist. [m]"])].apply(parse_float)
    tmp_raw["_HI_m"] = tmp_raw[config["hi_col"]].apply(parse_float)
    tmp_raw["_HT_m"] = tmp_raw[find_col(df_raw, ["Target Height [m]"])].apply(parse_float)

    # keep only fields needed
    raw_key_cols = ["_from","_to","_face","_V_gon","_slope_m","_HI_m","_HT_m"]
    raw_cmp = tmp_raw[raw_key_cols].copy()
    if col_vdist_raw is not None:
        raw_cmp["V_dist_raw_m"] = tmp_raw[col_vdist_raw].apply(parse_float)
    if col_dh_raw is not None:
        raw_cmp["dH_raw_m"] = tmp_raw[col_dh_raw].apply(parse_float)

    obs_key = obs.copy()
    obs_key["_from"] = obs_key["from"]
    obs_key["_to"] = obs_key["to"]
    obs_key["_face"] = obs_key["face"]
    obs_key["_V_gon"] = obs_key["V_gon"]
    obs_key["_slope_m"] = obs_key["slope_m"]
    obs_key["_HI_m"] = obs_key["HI_m"]
    obs_key["_HT_m"] = obs_key["HT_m"]

    merged = obs_key.merge(raw_cmp, on=raw_key_cols, how="left")

    if col_vdist_raw is not None:
        dv = merged["V_distC_m"] - merged["V_dist_raw_m"]
        dv = dv[np.isfinite(dv)]
        if len(dv) > 0:
            compare_report["V_distC_minus_raw"] = {
                "n": int(len(dv)),
                "mean_m": float(np.mean(dv)),
                "std_m": float(np.std(dv, ddof=1)) if len(dv) > 1 else 0.0,
                "min_m": float(np.min(dv)),
                "max_m": float(np.max(dv)),
            }

    if col_dh_raw is not None:
        ddh = merged["dH_C_m"] - merged["dH_raw_m"]
        ddh = ddh[np.isfinite(ddh)]
        if len(ddh) > 0:
            compare_report["dH_C_minus_raw"] = {
                "n": int(len(ddh)),
                "mean_m": float(np.mean(ddh)),
                "std_m": float(np.std(ddh, ddof=1)) if len(ddh) > 1 else 0.0,
                "min_m": float(np.min(ddh)),
                "max_m": float(np.max(ddh)),
            }

# --- Print summary ---
print("Computed columns added to `obs`: Z_rad, V_distC_m, dH_C_m")
print("\nBasic stats:")
print(obs[["V_gon","slope_m","V_distC_m","dH_C_m"]].describe())

if compare_report:
    print("\nComparison against raw-file columns (computed - raw):")
    for k, v in compare_report.items():
        print(f"\n{k}")
        for kk, vv in v.items():
            print(f"  {kk}: {vv}")
else:
    print("\nNo raw vertical-distance / ΔHeight columns found (or no comparable matches).")

obs.head(10)


Computed columns added to `obs`: Z_rad, V_distC_m, dH_C_m

Basic stats:
            V_gon     slope_m   V_distC_m      dH_C_m
count  278.000000  278.000000  278.000000  278.000000
mean   198.620060   38.080038    0.352594    1.840347
std    100.008670   29.704203    4.437117    4.443047
min     94.011500   16.243400  -10.009752   -8.590792
25%     98.878425   17.268900   -1.885826   -0.392966
50%    108.336800   21.365350    0.304277    1.897137
75%    301.118900   76.329900    1.099783    2.518743
max    305.986700   93.232300    7.169570    8.588530

Comparison against raw-file columns (computed - raw):

V_distC_minus_raw
  n: 296
  mean_m: -0.00015775431358508733
  std_m: 0.00021298152688878502
  min_m: -0.0006914541955991282
  max_m: 3.70273867177362e-05

dH_C_minus_raw
  n: 296
  mean_m: -0.0051730921514229625
  std_m: 0.002805947537004267
  min_m: -0.009663820116535327
  max_m: -0.0030109140998895167


,from,to,face,V_gon,slope_m,HI_m,HT_m,datetime,is_valid,Z_rad,V_distC_m,dH_C_m
0,R1,R2,I,108.0436,21.6415,1.51896,0.1,2025-01-19,True,1.697145,-2.727103,-1.308143
1,R1,R2,I,108.0442,21.6415,1.51896,0.1,2025-01-19,True,1.697154,-2.727306,-1.308346
2,R1,R2,I,108.0444,21.6408,1.51896,0.1,2025-01-19,True,1.697157,-2.727285,-1.308325
3,R1,R2,I,108.0454,21.6420,1.51896,0.1,2025-01-19,True,1.697173,-2.727773,-1.308813
4,R1,R2,I,108.0438,21.6422,1.51896,0.1,2025-01-19,True,1.697148,-2.727259,-1.308299
5,R1,R2,I,108.0437,21.6422,1.51896,0.1,2025-01-19,True,1.697146,-2.727225,-1.308265
6,R1,R2,I,108.0440,21.6420,1.51896,0.1,2025-01-19,True,1.697151,-2.727301,-1.308341
7,R1,R2,I,108.0447,21.6421,1.51896,0.1,2025-01-19,True,1.697162,-2.727550,-1.308590
8,R1,R2,I,108.0442,21.6421,1.51896,0.1,2025-01-19,True,1.697154,-2.727381,-1.308421
9,R1,R2,I,108.0440,21.6422,1.51896,0.1,2025-01-19,True,1.697151,-2.727326,-1.308366


In [ ]:
# === Network Coverage Report (Nodes & Edges) ===
# Name: network_coverage_report
# Description:
# - List unique benchmarks (nodes)
# - Build directed edge table: i->j with counts (total, Face I, Face II)
# - Build undirected edge table: {i,j} with counts
# - Identify reciprocal pairs (both directions present)
# - Provide compact coverage report tables

import pandas as pd
import numpy as np

# --- Nodes ---
nodes = sorted(set(obs["from"]).union(set(obs["to"])))
df_nodes = pd.DataFrame({"benchmark": nodes})
print(f"Benchmarks found (N={len(nodes)}): {nodes}")

# --- Directed edges with counts by face ---
edge_counts = (
    obs.groupby(["from", "to"])
       .agg(
           n_total=("dH_C_m", "size"),
           n_face_I=("face", lambda s: int((s == "I").sum())),
           n_face_II=("face", lambda s: int((s == "II").sum())),
           dH_mean=("dH_C_m", "mean"),
           dH_std=("dH_C_m", lambda s: float(np.std(s, ddof=1)) if len(s) > 1 else 0.0),
       )
       .reset_index()
       .sort_values(["from", "to"])
       .reset_index(drop=True)
)

print("\nDirected edges (from -> to):")
display(edge_counts)

# --- Undirected edges (pair key independent of direction) ---
tmp = obs.copy()
tmp["u"] = tmp[["from", "to"]].min(axis=1)
tmp["v"] = tmp[["from", "to"]].max(axis=1)

undir_counts = (
    tmp.groupby(["u", "v"])
       .agg(
           n_total=("dH_C_m", "size"),
           n_face_I=("face", lambda s: int((s == "I").sum())),
           n_face_II=("face", lambda s: int((s == "II").sum())),
       )
       .reset_index()
       .sort_values(["u", "v"])
       .reset_index(drop=True)
)

print("\nUndirected edges ({u, v}):")
display(undir_counts)

# --- Reciprocal pairs: both directions exist ---
directed_set = set(zip(edge_counts["from"], edge_counts["to"]))
recip_rows = []
for a, b in sorted({tuple(sorted([f, t])) for f, t in directed_set if f != t}):
    has_ab = (a, b) in directed_set
    has_ba = (b, a) in directed_set
    if has_ab and has_ba:
        # pull counts
        ab = edge_counts[(edge_counts["from"] == a) & (edge_counts["to"] == b)].iloc[0]
        ba = edge_counts[(edge_counts["from"] == b) & (edge_counts["to"] == a)].iloc[0]
        recip_rows.append({
            "pair": f"{a}<->{b}",
            "n_ab": int(ab["n_total"]),
            "n_ba": int(ba["n_total"]),
            "n_ab_I": int(ab["n_face_I"]),
            "n_ab_II": int(ab["n_face_II"]),
            "n_ba_I": int(ba["n_face_I"]),
            "n_ba_II": int(ba["n_face_II"]),
        })

df_recip = pd.DataFrame(recip_rows).sort_values("pair").reset_index(drop=True)

print("\nReciprocal pairs (i <-> j):")
if len(df_recip) == 0:
    print("None found.")
else:
    display(df_recip)

# Save for next cells
network = {
    "nodes": nodes,
    "edge_counts_directed": edge_counts,
    "edge_counts_undirected": undir_counts,
    "reciprocal_pairs": df_recip
}
print("\nSaved network summary in dict: network")


Benchmarks found (N=10): ['F1', 'F2', 'R1', 'R2', 'R3', 'R4', 'R5', 'T1', 'T2', 'T3']

Directed edges (from -> to):


,from,to,n_total,n_face_I,n_face_II,dH_mean,dH_std
0,R1,R2,27,13,14,-1.308544,0.000301
1,R1,R5,23,13,10,-8.589881,0.000601
2,R4,F1,12,6,6,2.155086,0.000375
3,R4,F2,12,6,6,2.130700,0.000399
4,R4,R1,22,11,11,8.198792,0.001094
5,R4,R5,25,13,12,-0.393171,0.000282
6,R4,T1,22,11,11,1.897853,0.000359
7,R4,T2,22,11,11,1.896561,0.000418
8,R4,T3,22,11,11,1.896692,0.000343
9,R5,R1,29,15,14,8.585798,0.001110



Undirected edges ({u, v}):


,u,v,n_total,n_face_I,n_face_II
0,F1,R4,12,6,6
1,F2,R4,12,6,6
2,R1,R2,27,13,14
3,R1,R4,22,11,11
4,R1,R5,52,28,24
5,R2,R5,10,5,5
6,R3,R5,26,13,13
7,R4,R5,51,26,25
8,R4,T1,22,11,11
9,R4,T2,22,11,11



Reciprocal pairs (i <-> j):


,pair,n_ab,n_ba,n_ab_I,n_ab_II,n_ba_I,n_ba_II
0,R1<->R5,23,29,13,10,15,14
1,R4<->R5,25,26,13,12,13,13



Saved network summary in dict: network


In [ ]:
# === Cell 6A — Directed-edge precision + Directed-edge QC (k·σ) ===
# Title (EN): Directed-Edge Internal Precision + QC
# Name: cell6A_directed_edge_precision_and_qc
# Description:
# - Computes Table I: directed-edge mean/std/SE for dH_C_m (overall + per face)
# - Optional QC per directed edge:
#     * mode="combined": per (from->to) using both faces together
#     * mode="per_face": per (from->to->face) using each face separately
# - Flags |dH - mean(sample)| > k*std(sample)
# - Optionally drops flagged observations -> creates obs_qc
# - Recomputes Table I on obs_qc -> edge_precision_clean
# - Saves: edge_precision, edge_precision_clean, flagged_edge_outliers, flagged_edge_summary,
#          k_used_edge_qc, qc_mode_edge_qc, obs_qc

import numpy as np
import pandas as pd

# ----------------------------
# Helpers (same behavior as original)
# ----------------------------
def _std_ddof1(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) <= 1:
        return 0.0
    return float(np.std(x, ddof=1))

def _se_from_std(std, n):
    return float(std / np.sqrt(n)) if n > 0 else np.nan

def _compute_table_I(obs_df):
    rows = []
    for (f, t), g in obs_df.groupby(["from", "to"]):
        vals_all = g["dH_C_m"].values
        n_all = len(vals_all)
        mean_all = float(np.mean(vals_all))
        std_all = _std_ddof1(vals_all)
        se_all = _se_from_std(std_all, n_all)

        gI = g[g["face"] == "I"]["dH_C_m"].values
        gII = g[g["face"] == "II"]["dH_C_m"].values

        nI, nII = len(gI), len(gII)
        meanI = float(np.mean(gI)) if nI > 0 else np.nan
        meanII = float(np.mean(gII)) if nII > 0 else np.nan
        stdI = _std_ddof1(gI) if nI > 0 else np.nan
        stdII = _std_ddof1(gII) if nII > 0 else np.nan
        seI = _se_from_std(stdI, nI) if nI > 0 else np.nan
        seII = _se_from_std(stdII, nII) if nII > 0 else np.nan

        rows.append({
            "from": f, "to": t,
            "n_all": n_all, "mean_all_m": mean_all, "std_all_m": std_all, "se_all_m": se_all,
            "n_I": nI, "mean_I_m": meanI, "std_I_m": stdI, "se_I_m": seI,
            "n_II": nII, "mean_II_m": meanII, "std_II_m": stdII, "se_II_m": seII,
            "mean_I_minus_II_m": (meanI - meanII) if (np.isfinite(meanI) and np.isfinite(meanII)) else np.nan,
        })

    return pd.DataFrame(rows).sort_values(["from", "to"]).reset_index(drop=True)

# ----------------------------
# 1) Table I on original obs (same as original cell up to this point)
# ----------------------------
edge_precision = _compute_table_I(obs)

print("Directed-edge internal precision (overall + per face):")
display(edge_precision)

# ----------------------------
# 2) Directed-edge QC (k·σ) and optional dropping -> obs_qc
# ----------------------------
print("\n--- Directed-edge QC (optional) ---", flush=True)
apply_qc = input("Apply directed-edge QC (k·σ) on dH_C_m? (y/n): ").strip().lower()
if apply_qc not in ["y", "n"]:
    raise ValueError("Please answer with 'y' or 'n'.")

k_used_edge_qc = None
qc_mode_edge_qc = None
flagged_edge_outliers = pd.DataFrame()
flagged_edge_summary = pd.DataFrame()

# Default: no change
obs_qc = obs.copy()
edge_precision_clean = edge_precision.copy()

if apply_qc == "y":
    # NEW: choose sample definition
    qc_mode_edge_qc = input(
        "QC sample for each direction: 'combined' (faces I+II together) or 'per_face' (each face separately)? "
    ).strip().lower()

    if qc_mode_edge_qc not in ["combined", "per_face"]:
        raise ValueError("Invalid choice. Use 'combined' or 'per_face'.")

    k_used_edge_qc = float(
        input("Enter k for |dH - mean(sample)| > k*std(sample) (e.g., 3): ").strip().replace(",", ".")
    )

    # Create stable row id to drop exactly those rows
    obs_work = obs.copy().reset_index().rename(columns={"index": "obs_id"})

    flag_rows = []

    # Grouping for QC
    if qc_mode_edge_qc == "combined":
        group_cols = ["from", "to"]
    else:  # per_face
        group_cols = ["from", "to", "face"]

    for gkey, g in obs_work.groupby(group_cols):
        vals = g["dH_C_m"].values.astype(float)
        mu = float(np.mean(vals))
        sig = _std_ddof1(vals)

        if sig == 0.0:
            is_out = np.zeros(len(g), dtype=bool)
        else:
            is_out = np.abs(vals - mu) > k_used_edge_qc * sig

        if np.any(is_out):
            tmp = g.loc[is_out, ["obs_id","from","to","face","datetime","V_gon","slope_m","HI_m","HT_m","dH_C_m","V_distC_m"]].copy()
            tmp["sample_mean_m"] = mu
            tmp["sample_std_m"] = sig
            tmp["residual_m"] = tmp["dH_C_m"] - mu
            tmp["k_used"] = k_used_edge_qc
            tmp["qc_mode"] = qc_mode_edge_qc
            flag_rows.append(tmp)

    flagged_edge_outliers = pd.concat(flag_rows, ignore_index=True) if flag_rows else pd.DataFrame()

    # Summary
    if len(flagged_edge_outliers) > 0:
        if qc_mode_edge_qc == "combined":
            flagged_edge_summary = (
                flagged_edge_outliers.groupby(["from","to"])
                .size()
                .reset_index(name="n_flagged")
                .sort_values(["from","to"])
                .reset_index(drop=True)
            )
        else:
            flagged_edge_summary = (
                flagged_edge_outliers.groupby(["from","to","face"])
                .size()
                .reset_index(name="n_flagged")
                .sort_values(["from","to","face"])
                .reset_index(drop=True)
            )
    else:
        # define empty summary with expected columns
        flagged_edge_summary = pd.DataFrame(
            columns=(["from","to","n_flagged"] if qc_mode_edge_qc == "combined" else ["from","to","face","n_flagged"])
        )

    print("\nFlagged outliers per sample:", flush=True)
    display(flagged_edge_summary)

    if len(flagged_edge_outliers) > 0:
        print("\nFlagged observations (first 50 rows):", flush=True)
        display(flagged_edge_outliers.head(50))
    else:
        print("\nNo observations flagged with the chosen k.", flush=True)

    drop_flagged = input("Drop flagged observations and recompute Table I? (y/n): ").strip().lower()
    if drop_flagged not in ["y", "n"]:
        raise ValueError("Please answer with 'y' or 'n'.")

    if drop_flagged == "y" and len(flagged_edge_outliers) > 0:
        drop_ids = set(flagged_edge_outliers["obs_id"].tolist())
        before = len(obs_work)
        obs_qc = obs_work[~obs_work["obs_id"].isin(drop_ids)].drop(columns=["obs_id"]).copy()
        after = len(obs_qc)
        print(f"\nDropped {before-after} rows. Remaining: {after}", flush=True)
    else:
        obs_qc = obs.copy()
        print("\nNo rows dropped. Using original obs as obs_qc.", flush=True)

    # Recompute Table I on obs_qc
    edge_precision_clean = _compute_table_I(obs_qc)
    print("\nDirected-edge internal precision (overall + per face) AFTER directed-edge QC:")
    display(edge_precision_clean)

# Save for next cell
qc_edge = {
    "edge_precision": edge_precision,                    # Table I (original)
    "edge_precision_clean": edge_precision_clean,        # Table I after QC (may equal original)
    "flagged_edge_outliers": flagged_edge_outliers,      # detailed flagged rows (if QC enabled)
    "flagged_edge_summary": flagged_edge_summary,        # counts per sample (if QC enabled)
    "k_used_edge_qc": k_used_edge_qc,                    # k used (if enabled)
    "qc_mode_edge_qc": qc_mode_edge_qc,                  # combined / per_face (if enabled)
    "obs_qc": obs_qc                                     # observations after QC (or original)
}
print("\nSaved directed-edge QC results in dict: qc_edge")
print("Key output for next cell: obs_qc")


Directed-edge internal precision (overall + per face):


,from,to,n_all,mean_all_m,std_all_m,se_all_m,n_I,mean_I_m,std_I_m,se_I_m,n_II,mean_II_m,std_II_m,se_II_m,mean_I_minus_II_m
0,R1,R2,27,-1.308544,0.000301,0.000058,13,-1.308388,0.000178,0.000049,14,-1.308689,0.000324,0.000087,0.000301
1,R1,R5,23,-8.589881,0.000601,0.000125,13,-8.589586,0.000438,0.000122,10,-8.590266,0.000580,0.000184,0.000680
2,R4,F1,12,2.155086,0.000375,0.000108,6,2.155363,0.000263,0.000108,6,2.154808,0.000234,0.000095,0.000555
3,R4,F2,12,2.130700,0.000399,0.000115,6,2.131029,0.000207,0.000085,6,2.130371,0.000217,0.000088,0.000659
4,R4,R1,22,8.198792,0.001094,0.000233,11,8.199306,0.000563,0.000170,11,8.198278,0.001271,0.000383,0.001028
5,R4,R5,25,-0.393171,0.000282,0.000056,13,-0.392964,0.000177,0.000049,12,-0.393395,0.000184,0.000053,0.000431
6,R4,T1,22,1.897853,0.000359,0.000077,11,1.898143,0.000196,0.000059,11,1.897562,0.000216,0.000065,0.000581
7,R4,T2,22,1.896561,0.000418,0.000089,11,1.896914,0.000244,0.000074,11,1.896208,0.000183,0.000055,0.000707
8,R4,T3,22,1.896692,0.000343,0.000073,11,1.896973,0.000244,0.000074,11,1.896411,0.000119,0.000036,0.000562
9,R5,R1,29,8.585798,0.001110,0.000206,15,8.586460,0.001075,0.000278,14,8.585088,0.000599,0.000160,0.001371



--- Directed-edge QC (optional) ---
Apply directed-edge QC (k·σ) on dH_C_m? (y/n): y
QC sample for each direction: 'combined' (faces I+II together) or 'per_face' (each face separately)? per_face
Enter k for |dH - mean(sample)| > k*std(sample) (e.g., 3): 2

Flagged outliers per sample:


,from,to,face,n_flagged
0,R1,R2,I,1
1,R1,R2,II,1
2,R4,R1,I,1
3,R4,R1,II,1
4,R4,R5,I,1
5,R5,R1,II,1
6,R5,R3,II,1



Flagged observations (first 50 rows):


,obs_id,from,to,face,datetime,V_gon,slope_m,HI_m,HT_m,dH_C_m,V_distC_m,sample_mean_m,sample_std_m,residual_m,k_used,qc_mode
0,3,R1,R2,I,2025-01-19,108.0454,21.6420,1.51896,0.1,-1.308813,-2.727773,-1.308388,0.000178,-0.000425,2.0,per_face
1,15,R1,R2,II,2025-01-19,291.9526,21.6438,1.51896,0.1,-1.309715,-2.728675,-1.308689,0.000324,-0.001025,2.0,per_face
2,74,R4,R1,I,2025-01-19,95.4157,93.2321,1.59286,0.1,8.200704,6.707844,8.199306,0.000563,0.001398,2.0,per_face
3,86,R4,R1,II,2025-01-19,304.5852,93.2320,1.59286,0.1,8.202012,6.709152,8.198278,0.001271,0.003734,2.0,per_face
4,105,R4,R5,I,2025-01-19,105.6104,21.4219,1.59286,0.1,-0.392566,-1.885426,-0.392964,0.000177,0.000398,2.0,per_face
5,205,R5,R1,II,2025-01-19,305.9867,76.3298,1.51896,0.1,8.586353,7.167393,8.585088,0.000599,0.001264,2.0,per_face
6,249,R5,R3,II,2025-01-19,304.3100,16.2435,1.51896,0.1,2.517827,1.098867,2.518082,0.000097,-0.000256,2.0,per_face


Drop flagged observations and recompute Table I? (y/n): y

Dropped 7 rows. Remaining: 271

Directed-edge internal precision (overall + per face) AFTER directed-edge QC:


,from,to,n_all,mean_all_m,std_all_m,se_all_m,n_I,mean_I_m,std_I_m,se_I_m,n_II,mean_II_m,std_II_m,se_II_m,mean_I_minus_II_m
0,R1,R2,25,-1.308486,0.000186,0.000037,12,-1.308352,0.000130,0.000038,13,-1.308610,0.000139,0.000039,0.000258
1,R1,R5,23,-8.589881,0.000601,0.000125,13,-8.589586,0.000438,0.000122,10,-8.590266,0.000580,0.000184,0.000680
2,R4,F1,12,2.155086,0.000375,0.000108,6,2.155363,0.000263,0.000108,6,2.154808,0.000234,0.000095,0.000555
3,R4,F2,12,2.130700,0.000399,0.000115,6,2.131029,0.000207,0.000085,6,2.130371,0.000217,0.000088,0.000659
4,R4,R1,20,8.198535,0.000718,0.000161,10,8.199166,0.000337,0.000107,10,8.197904,0.000300,0.000095,0.001262
5,R4,R5,24,-0.393196,0.000258,0.000053,12,-0.392997,0.000136,0.000039,12,-0.393395,0.000184,0.000053,0.000398
6,R4,T1,22,1.897853,0.000359,0.000077,11,1.898143,0.000196,0.000059,11,1.897562,0.000216,0.000065,0.000581
7,R4,T2,22,1.896561,0.000418,0.000089,11,1.896914,0.000244,0.000074,11,1.896208,0.000183,0.000055,0.000707
8,R4,T3,22,1.896692,0.000343,0.000073,11,1.896973,0.000244,0.000074,11,1.896411,0.000119,0.000036,0.000562
9,R5,R1,28,8.585778,0.001125,0.000213,15,8.586460,0.001075,0.000278,13,8.584991,0.000496,0.000137,0.001468



Saved directed-edge QC results in dict: qc_edge
Key output for next cell: obs_qc


In [ ]:
# === Cell 6Bi Reciprocal Misclosure Summary (No Inputs) ===
# Name: reciprocal_misclosure_summary_only
# Description:
# - Compute and display: Reciprocal misclosure summary (m = dH(i->j) + dH(j->i))
# - No user prompts in this cell (always terminates)

import numpy as np
import pandas as pd

# ---------- Helpers (same as original style) ----------
def _std_ddof1(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) <= 1:
        return 0.0
    return float(np.std(x, ddof=1))

def reciprocal_misclosures(obs_df, a, b):
    """
    Return DataFrame of misclosures for pair a<->b:
    m = dH(a->b) + dH(b->a)
    With date-only (single value), use all-by-all combinations.
    """
    ab = obs_df[(obs_df["from"] == a) & (obs_df["to"] == b)].copy()
    ba = obs_df[(obs_df["from"] == b) & (obs_df["to"] == a)].copy()

    if len(ab) == 0 or len(ba) == 0:
        return pd.DataFrame(columns=["a","b","m_m","dH_ab_m","dH_ba_m","face_ab","face_ba"])

    ab_small = ab[["dH_C_m","face"]].rename(columns={"dH_C_m":"dH_ab_m","face":"face_ab"})
    ba_small = ba[["dH_C_m","face"]].rename(columns={"dH_C_m":"dH_ba_m","face":"face_ba"})

    ab_small["_k"] = 1
    ba_small["_k"] = 1
    mdf = ab_small.merge(ba_small, on="_k", how="outer").drop(columns="_k")
    mdf["m_m"] = mdf["dH_ab_m"] + mdf["dH_ba_m"]
    mdf.insert(0, "a", a)
    mdf.insert(1, "b", b)
    return mdf

# ---------- Choose observations (prefer QC output from Cell 6A) ----------
if "qc_edge" in globals() and isinstance(qc_edge, dict) and isinstance(qc_edge.get("obs_qc", None), pd.DataFrame) and len(qc_edge["obs_qc"]) > 0:
    obs_used = qc_edge["obs_qc"]
elif "obs_qc" in globals() and isinstance(obs_qc, pd.DataFrame) and len(obs_qc) > 0:
    obs_used = obs_qc
else:
    obs_used = obs

# Reciprocal pairs from network
recip_pairs = network["reciprocal_pairs"]["pair"].tolist() if len(network["reciprocal_pairs"]) else []

recip_summary_rows = []
recip_m_all = []

for p in recip_pairs:
    a, b = p.split("<->")
    mdf = reciprocal_misclosures(obs_used, a, b)
    if len(mdf) == 0:
        continue

    mvals = mdf["m_m"].values
    mvals = mvals[np.isfinite(mvals)]

    recip_m_all.append(mdf)

    recip_summary_rows.append({
        "pair": p,
        "n_misclosures": int(len(mvals)),
        "mean_m_m": float(np.mean(mvals)),
        "std_m_m": _std_ddof1(mvals),
        "min_m_m": float(np.min(mvals)),
        "max_m_m": float(np.max(mvals)),
    })

recip_summary = pd.DataFrame(recip_summary_rows).sort_values("pair").reset_index(drop=True)

print("\nReciprocal misclosure summary (m = dH(i->j) + dH(j->i)):")
display(recip_summary)

# Save for next cell
qc_rec = {
    "obs_used": obs_used,
    "recip_pairs": recip_pairs,
    "recip_m_all": recip_m_all,
    "reciprocal_summary": recip_summary
}
print("\nSaved reciprocal results in dict: qc_rec")



Reciprocal misclosure summary (m = dH(i->j) + dH(j->i)):


,pair,n_misclosures,mean_m_m,std_m_m,min_m_m,max_m_m
0,R1<->R5,644,-0.004104,0.001252,-0.006569,-0.000232
1,R4<->R5,624,-0.004288,0.000376,-0.005310,-0.003393



Saved reciprocal results in dict: qc_rec


In [ ]:
# === Cell 6Bii (v2) Reciprocal Outlier Flagging + obs_id tracing ===
# Name: reciprocal_outlier_flagging_with_ids
# Description:
# - Prompts: none / abs / ksig
# - Flags outliers on reciprocal misclosures
# - Prints flagged combinations INCLUDING obs_id_ab, obs_id_ba so you can trace back to obs_used
# - Saves results in dict: qc_rec_flags

import numpy as np
import pandas as pd

# ---------- Helpers ----------
def _std_ddof1(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) <= 1:
        return 0.0
    return float(np.std(x, ddof=1))

def reciprocal_misclosures_with_ids(obs_df, a, b):
    """
    Return DataFrame of misclosures for pair a<->b:
    m = dH(a->b) + dH(b->a)
    Uses all-by-all combinations (cartesian product).
    Includes obs_id_ab / obs_id_ba for tracing.
    """
    # Create stable obs_id within current obs_df
    obs_id_df = obs_df.reset_index(drop=True).copy()
    obs_id_df["obs_id"] = np.arange(len(obs_id_df), dtype=int)

    ab = obs_id_df[(obs_id_df["from"] == a) & (obs_id_df["to"] == b)].copy()
    ba = obs_id_df[(obs_id_df["from"] == b) & (obs_id_df["to"] == a)].copy()

    if len(ab) == 0 or len(ba) == 0:
        return pd.DataFrame(columns=[
            "a","b","m_m",
            "obs_id_ab","dH_ab_m","face_ab","V_gon_ab","slope_m_ab",
            "obs_id_ba","dH_ba_m","face_ba","V_gon_ba","slope_m_ba"
        ])

    ab_small = ab[["obs_id","dH_C_m","face","V_gon","slope_m"]].rename(columns={
        "obs_id":"obs_id_ab", "dH_C_m":"dH_ab_m", "face":"face_ab",
        "V_gon":"V_gon_ab", "slope_m":"slope_m_ab"
    })
    ba_small = ba[["obs_id","dH_C_m","face","V_gon","slope_m"]].rename(columns={
        "obs_id":"obs_id_ba", "dH_C_m":"dH_ba_m", "face":"face_ba",
        "V_gon":"V_gon_ba", "slope_m":"slope_m_ba"
    })

    ab_small["_k"] = 1
    ba_small["_k"] = 1
    mdf = ab_small.merge(ba_small, on="_k", how="outer").drop(columns="_k")

    mdf["m_m"] = mdf["dH_ab_m"] + mdf["dH_ba_m"]
    mdf.insert(0, "a", a)
    mdf.insert(1, "b", b)
    return mdf

# ---------- Load what 6Bi computed ----------
if "qc_rec" not in globals():
    raise RuntimeError("qc_rec not found. Run Cell 6Bi first.")

# Prefer latest directed-edge QC output if available (stronger upstream source)
if "qc_edge" in globals() and isinstance(qc_edge, dict) and isinstance(qc_edge.get("obs_qc", None), pd.DataFrame) and len(qc_edge["obs_qc"]) > 0:
    obs_used = qc_edge["obs_qc"]
else:
    obs_used = qc_rec["obs_used"]

recip_pairs = qc_rec["recip_pairs"]

# ---------- PROMPTS ----------
mode = input("Outlier flagging for misclosures? Choose: 'none' / 'abs' / 'ksig' : ").strip().lower()
if mode not in ["none", "abs", "ksig"]:
    raise ValueError("Invalid choice. Use 'none', 'abs', or 'ksig'.")

thr_m = None
k = None

if mode == "abs":
    thr_mm = float(input("Absolute threshold in mm (e.g., 5): ").strip().replace(",", "."))
    thr_m = thr_mm / 1000.0
elif mode == "ksig":
    k = float(input("k for |m - mean(m)| > k*std(m) (e.g., 3): ").strip().replace(",", "."))

# ---------- Apply rule ----------
flagged = None

if mode != "none" and recip_pairs:
    all_flag_rows = []

    for p in recip_pairs:
        a, b = p.split("<->")
        mdf = reciprocal_misclosures_with_ids(obs_used, a, b).copy()

        mvals = mdf["m_m"].values
        mvals = mvals[np.isfinite(mvals)]
        if len(mvals) == 0:
            continue

        mu = float(np.mean(mvals))
        sig = _std_ddof1(mvals)

        if mode == "abs":
            mdf["is_outlier"] = np.abs(mdf["m_m"].values) > thr_m
        elif mode == "ksig":
            if sig == 0.0:
                mdf["is_outlier"] = False
            else:
                mdf["is_outlier"] = np.abs(mdf["m_m"].values - mu) > k * sig

        out = mdf[mdf["is_outlier"]].copy()
        out["pair"] = p
        out["mean_m_pair_m"] = mu
        out["std_m_pair_m"] = sig
        all_flag_rows.append(out)

    if all_flag_rows:
        flagged = pd.concat(all_flag_rows, ignore_index=True)

        # Keep compact view, but with obs ids + some context
        flagged = flagged[[
            "pair","m_m",
            "obs_id_ab","dH_ab_m","face_ab","V_gon_ab","slope_m_ab",
            "obs_id_ba","dH_ba_m","face_ba","V_gon_ba","slope_m_ba",
            "mean_m_pair_m","std_m_pair_m","is_outlier"
        ]]

        print("\nFlagged misclosure combinations (with obs_id tracing):")
        display(flagged.head(50))

        print("\nHow to trace back:")
        print(" - obs_id_ab refers to row in obs_used.reset_index(drop=True)")
        print(" - obs_id_ba refers to row in obs_used.reset_index(drop=True)")
        print("Example:")
        print("   obs_used.reset_index(drop=True).loc[obs_id_ab]")
        print("   obs_used.reset_index(drop=True).loc[obs_id_ba]")
    else:
        print("\nNo outliers flagged under the selected rule.")

# ---------- Save ----------
qc_rec_flags = {
    "mode": mode,
    "k": k,
    "thr_m": thr_m,
    "flagged_misclosures": flagged
}
print("\nSaved reciprocal flagging results in dict: qc_rec_flags")


Outlier flagging for misclosures? Choose: 'none' / 'abs' / 'ksig' : ksig
k for |m - mean(m)| > k*std(m) (e.g., 3): 3

Flagged misclosure combinations (with obs_id tracing):


,pair,m_m,obs_id_ab,dH_ab_m,face_ab,V_gon_ab,slope_m_ab,obs_id_ba,dH_ba_m,face_ba,V_gon_ba,slope_m_ba,mean_m_pair_m,std_m_pair_m,is_outlier
0,R1<->R5,-0.000232,27,-8.588761,I,108.3356,76.6514,185,8.588530,I,94.0115,76.3301,-0.004104,0.001252,True
1,R1<->R5,-0.000260,27,-8.588761,I,108.3356,76.6514,193,8.588501,I,94.0115,76.3298,-0.004104,0.001252,True



How to trace back:
 - obs_id_ab refers to row in obs_used.reset_index(drop=True)
 - obs_id_ba refers to row in obs_used.reset_index(drop=True)
Example:
   obs_used.reset_index(drop=True).loc[obs_id_ab]
   obs_used.reset_index(drop=True).loc[obs_id_ba]

Saved reciprocal flagging results in dict: qc_rec_flags


In [ ]:
# === CELL 7 — Least Squares Adjustment (Hard-Constraint Datum) ===
# Title (EN): Weighted LS Adjustment of Heights (Hard Constraints)
# Name: cell7_lsq_adjustment_heights
# Strategy:
# - Observations: mean dH per directed edge (from->to) per face separately
# - Weights: normalized W(I), normalized W(II) per WEIGHTS_IDEA_UPDATED logic:
#     A = sqrt(|meanI-meanII| / min(|meanI-meanII|))  (with clamping)
#     B(face) = sqrt(std_face)
#     W(face) = sqrt(1/(A*B(face)))
#     normalized W = W / max(W over all faces and edges)
# - Hard constraint: user-selected anchored benchmark with user-selected height
#
# Requires:
# - obs (and preferably obs_qc via qc_edge["obs_qc"])
# - columns in obs/obs_qc: ['from','to','face','dH_C_m']

import numpy as np
import pandas as pd

# ----------------------------
# 0) Select observations
# ----------------------------
if "qc_edge" in globals() and isinstance(qc_edge, dict) and isinstance(qc_edge.get("obs_qc", None), pd.DataFrame) and len(qc_edge["obs_qc"]) > 0:
    obs_used = qc_edge["obs_qc"].copy()
elif "obs_qc" in globals() and isinstance(obs_qc, pd.DataFrame) and len(obs_qc) > 0:
    obs_used = obs_qc.copy()
else:
    obs_used = obs.copy()

obs_used = obs_used.copy()
obs_used = obs_used[np.isfinite(obs_used["dH_C_m"].astype(float))].copy()

# ----------------------------
# 1) Helper stats
# ----------------------------
def _std_ddof1(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) <= 1:
        return 0.0
    return float(np.std(x, ddof=1))

# ----------------------------
# 2) Build directed-edge per-face observation table
# ----------------------------
rows = []
for (f, t, face), g in obs_used.groupby(["from","to","face"]):
    vals = g["dH_C_m"].astype(float).values
    rows.append({
        "from": f,
        "to": t,
        "face": face,
        "n": int(len(vals)),
        "dH_mean_m": float(np.mean(vals)),
        "dH_std_m": _std_ddof1(vals),
    })

obs_face = pd.DataFrame(rows).sort_values(["from","to","face"]).reset_index(drop=True)

print("LS Observations (mean dH per from->to per face):")
display(obs_face)

# ----------------------------
# 3) Compute A and weights per face (WEIGHTS_IDEA_UPDATED)
# ----------------------------
pivot = obs_face.pivot_table(
    index=["from","to"],
    columns="face",
    values=["dH_mean_m","dH_std_m","n"],
    aggfunc="first"
)

pivot.columns = [f"{a}_{b}" for a,b in pivot.columns]
pivot = pivot.reset_index()

for col in ["dH_mean_m_I","dH_mean_m_II","dH_std_m_I","dH_std_m_II"]:
    if col not in pivot.columns:
        pivot[col] = np.nan

pivot["abs_mean_I_minus_II_m"] = np.abs(pivot["dH_mean_m_I"] - pivot["dH_mean_m_II"])

valid_deltas = pivot["abs_mean_I_minus_II_m"].values
valid_deltas = valid_deltas[np.isfinite(valid_deltas)]

if len(valid_deltas) == 0:
    delta_min = 0.001  # 1 mm fallback when no edges have both faces
    print("\nWARNING: No edges have both faces to compute |meanI-meanII|. Using delta_min=1 mm fallback.")
else:
    delta_min_raw = float(np.min(valid_deltas))
    # Clamp: if too small (<0.1 mm) warn and set to 0.1 mm
    if delta_min_raw < 0.0001:
        print(f"\nWARNING: min(|meanI-meanII|) = {1000 * delta_min_raw:.6f} mm < 0.1 mm. Clamping delta_min to 0.1 mm.")
        delta_min = 0.0001
    else:
        delta_min = delta_min_raw

pivot["A"] = np.sqrt(pivot["abs_mean_I_minus_II_m"] / delta_min)
pivot.loc[~np.isfinite(pivot["A"]), "A"] = 1.0

pivot["B_I"]  = np.sqrt(pivot["dH_std_m_I"].astype(float))
pivot["B_II"] = np.sqrt(pivot["dH_std_m_II"].astype(float))

B_floor = 1e-6
pivot["B_I"]  = pivot["B_I"].where(np.isfinite(pivot["B_I"]) & (pivot["B_I"] > B_floor), B_floor)
pivot["B_II"] = pivot["B_II"].where(np.isfinite(pivot["B_II"]) & (pivot["B_II"] > B_floor), B_floor)

pivot["W_I"]  = np.sqrt(1.0 / (pivot["A"] * pivot["B_I"]))
pivot["W_II"] = np.sqrt(1.0 / (pivot["A"] * pivot["B_II"]))

W_all = np.concatenate([pivot["W_I"].values, pivot["W_II"].values])
W_all = W_all[np.isfinite(W_all)]
Wmax = float(np.max(W_all)) if len(W_all) else 1.0
pivot["W_I_norm"]  = pivot["W_I"]  / Wmax
pivot["W_II_norm"] = pivot["W_II"] / Wmax

w_rows = []
for _, r in pivot.iterrows():
    w_rows.append({"from": r["from"], "to": r["to"], "face":"I",  "W_norm": float(r["W_I_norm"])})
    w_rows.append({"from": r["from"], "to": r["to"], "face":"II", "W_norm": float(r["W_II_norm"])})
wtab = pd.DataFrame(w_rows)

obs_face = obs_face.merge(wtab, on=["from","to","face"], how="left")

weight_power = 2
obs_face["P"] = (obs_face["W_norm"].astype(float) ** weight_power)

print("\nWeights summary (per directed edge, includes A, B, W_norm):")
display(pivot[["from","to","abs_mean_I_minus_II_m","A","dH_std_m_I","dH_std_m_II","W_I_norm","W_II_norm"]])

print("\nLS observation table with weights (P):")
display(obs_face)

# ----------------------------
# 4) Hard constraint: user selects anchored benchmark and height
# ----------------------------
all_bm = sorted(set(obs_face["from"].tolist() + obs_face["to"].tolist()))
if len(all_bm) == 0:
    raise RuntimeError("No benchmarks found in obs_face.")

print("\nAvailable benchmarks for anchoring:")
for i, b in enumerate(all_bm, start=1):
    print(f"  {i:2d}) {b}")

anch_idx = input("\nSelect anchored benchmark by number (e.g., 1): ").strip()
if not anch_idx.isdigit():
    raise ValueError("Anchor selection must be an integer index.")
anch_idx = int(anch_idx)
if anch_idx < 1 or anch_idx > len(all_bm):
    raise ValueError("Anchor index out of range.")

anchor_bm = all_bm[anch_idx - 1]

anchor_H_str = input(f"Enter anchored height H for {anchor_bm} in meters (e.g., 123.456): ").strip().replace(",", ".")
try:
    anchor_H = float(anchor_H_str)
except Exception:
    raise ValueError("Anchored height must be a valid float (meters).")

fixed = {str(anchor_bm): float(anchor_H)}
print(f"\nDatum hard constraint: {anchor_bm} = {anchor_H:.4f} m")

fixed_bms = sorted(fixed.keys())

all_bms = all_bm
unknown_bms = [b for b in all_bms if b not in fixed_bms]
if len(unknown_bms) == 0:
    raise RuntimeError("All benchmarks are fixed; nothing to adjust.")

idx = {bm:i for i, bm in enumerate(unknown_bms)}

# ----------------------------
# 5) Build LS system: A x ≈ l
# ----------------------------
m = len(obs_face)
n = len(unknown_bms)

A = np.zeros((m, n), dtype=float)
l = np.zeros((m, 1), dtype=float)
P = np.zeros((m, m), dtype=float)

for i, r in obs_face.reset_index(drop=True).iterrows():
    f = r["from"]; t = r["to"]
    dH = float(r["dH_mean_m"])
    w  = float(r["P"]) if np.isfinite(float(r["P"])) else 0.0

    rhs = dH

    if t in idx:
        A[i, idx[t]] += 1.0
    else:
        rhs -= fixed[t]

    if f in idx:
        A[i, idx[f]] += -1.0
    else:
        rhs += fixed[f]

    l[i, 0] = rhs
    P[i, i] = w

keep = np.diag(P) > 0
A = A[keep, :]
l = l[keep, :]
P = P[np.ix_(keep, keep)]
obs_face_used = obs_face.reset_index(drop=True).loc[keep].reset_index(drop=True).copy()

m_used = A.shape[0]
dof = m_used - n
if dof <= 0:
    raise RuntimeError(f"Not enough observations for adjustment: m={m_used}, n={n}, dof={dof}")

# ----------------------------
# 6) Solve weighted LS
# ----------------------------
N = A.T @ P @ A
u = A.T @ P @ l

Ninv = np.linalg.inv(N)
x = Ninv @ u

v = A @ x - l
sigma0_sq = float((v.T @ P @ v) / dof)
sigma0 = float(np.sqrt(sigma0_sq))

Cx = sigma0_sq * Ninv

# ----------------------------
# 7) Estimated heights + uncertainties
# ----------------------------
H_est = {}
H_sig = {}

for bm, H in fixed.items():
    H_est[bm] = float(H)
    H_sig[bm] = 0.0

for bm, j in idx.items():
    H_est[bm] = float(x[j,0])
    H_sig[bm] = float(np.sqrt(Cx[j,j]))

heights = pd.DataFrame({
    "benchmark": all_bms,
    "H_hat_m": [H_est[str(b)] for b in all_bms],
    "sigma_H_m": [H_sig[str(b)] for b in all_bms],
    "is_fixed": [str(b) in fixed_bms for b in all_bms]
}).sort_values("benchmark").reset_index(drop=True)

residuals = obs_face_used.copy()
residuals["v_m"] = v.reshape(-1)

print("\n--- LS Adjustment Results ---")
print(f"Used observations: m={m_used}, unknowns: n={n}, dof={dof}")
print(f"Posterior sigma0: {sigma0:.6e} (m)")

print("\nEstimated heights:")
display(heights)

print("\nResiduals (first 50):")
display(residuals.head(50))

# ----------------------------
# 8) Save outputs
# ----------------------------
lsq = {
    "obs_used": obs_used,
    "obs_face": obs_face,
    "obs_face_used": obs_face_used,
    "fixed": fixed,
    "unknowns": unknown_bms,
    "anchor_bm": anchor_bm,
    "anchor_H": anchor_H,
    "A": A, "l": l, "P": P,
    "x": x, "v": v,
    "sigma0": sigma0, "sigma0_sq": sigma0_sq,
    "Cx": Cx,
    "heights": heights,
    "residuals": residuals,
    "weights_details": pivot
}

print("\nSaved LS outputs in dict: lsq")


LS Observations (mean dH per from->to per face):


,from,to,face,n,dH_mean_m,dH_std_m
0,R1,R2,I,12,-1.308352,0.000130
1,R1,R2,II,13,-1.308610,0.000139
2,R1,R5,I,13,-8.589586,0.000438
3,R1,R5,II,10,-8.590266,0.000580
4,R4,F1,I,6,2.155363,0.000263
5,R4,F1,II,6,2.154808,0.000234
6,R4,F2,I,6,2.131029,0.000207
7,R4,F2,II,6,2.130371,0.000217
8,R4,R1,I,10,8.199166,0.000337
9,R4,R1,II,10,8.197904,0.000300



Weights summary (per directed edge, includes A, B, W_norm):


,from,to,abs_mean_I_minus_II_m,A,dH_std_m_I,dH_std_m_II,W_I_norm,W_II_norm
0,R1,R2,0.000258,1.000000,0.000130,0.000139,1.000000,0.983493
1,R1,R5,0.000680,1.623738,0.000438,0.000580,0.579203,0.539891
2,R4,F1,0.000555,1.467603,0.000263,0.000234,0.691877,0.712862
3,R4,F2,0.000659,1.598106,0.000207,0.000217,0.703904,0.696196
4,R4,R1,0.001262,2.212081,0.000337,0.000300,0.529854,0.545455
5,R4,R5,0.000398,1.241567,0.000136,0.000184,0.886664,0.822933
6,R4,T1,0.000581,1.501005,0.000196,0.000216,0.737088,0.718626
7,R4,T2,0.000707,1.655238,0.000244,0.000183,0.663885,0.714044
8,R4,T3,0.000562,1.476540,0.000244,0.000119,0.703055,0.841726
9,R5,R1,0.001468,2.386187,0.001075,0.000496,0.381749,0.463288



LS observation table with weights (P):


...

/tmp/ipython-input-646583702.py:222: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  sigma0_sq = float((v.T @ P @ v) / dof)


,benchmark,H_hat_m,sigma_H_m,is_fixed
0,F1,3.957760,0.001515,False
1,F2,3.933386,0.001518,False
2,R1,10.000000,0.000000,True
3,R2,8.691113,0.000786,False
4,R3,3.930150,0.001291,False
5,R4,1.802682,0.000972,False
6,R5,1.411783,0.000895,False
7,T1,3.700542,0.001484,False
8,T2,3.699218,0.001532,False
9,T3,3.699325,0.001433,False



Residuals (first 50):


,from,to,face,n,dH_mean_m,dH_std_m,W_norm,P,v_m
0,R1,R2,I,12,-1.308352,0.000130,1.000000,1.000000,-0.000534
1,R1,R2,II,13,-1.308610,0.000139,0.983493,0.967258,-0.000277
2,R1,R5,I,13,-8.589586,0.000438,0.579203,0.335476,0.001369
3,R1,R5,II,10,-8.590266,0.000580,0.539891,0.291482,0.002049
4,R4,F1,I,6,2.155363,0.000263,0.691877,0.478693,-0.000286
5,R4,F1,II,6,2.154808,0.000234,0.712862,0.508172,0.000269
6,R4,F2,I,6,2.131029,0.000207,0.703904,0.495481,-0.000326
7,R4,F2,II,6,2.130371,0.000217,0.696196,0.484689,0.000333
8,R4,R1,I,10,8.199166,0.000337,0.529854,0.280745,-0.001849
9,R4,R1,II,10,8.197904,0.000300,0.545455,0.297521,-0.000587



Saved LS outputs in dict: lsq


**TO DO**

In [ ]:
# === Input External Constraints (Anchors & Extra ΔH) ===
# Name: input_external_constraints
# Description:
# - Ask user if they have absolute heights for any benchmark(s) + their uncertainty (sigma)
# - Ask user if they have additional ΔH observations between benchmarks from other methods/campaigns + uncertainty (sigma)
# - Store in DataFrames: `anchors` and `external_dh`
# - Validate benchmark names against detected network nodes

import numpy as np
import pandas as pd

nodes = network["nodes"]
node_set = set(nodes)

import sys

def _ask_yes_no(prompt: str) -> bool:
    print(prompt, end="")
    sys.stdout.flush()
    ans = input().strip().lower()
    if ans not in ["y", "n"]:
        raise ValueError("Please answer with 'y' or 'n'.")
    return ans == "y"


def _read_int(prompt: str, min_val: int = 0) -> int:
    v = int(input(prompt).strip())
    if v < min_val:
        raise ValueError(f"Value must be >= {min_val}.")
    return v

def _read_float(prompt: str) -> float:
    s = input(prompt).strip().replace(",", ".")
    return float(s)

def _read_benchmark(prompt: str) -> str:
    b = input(prompt).strip()
    if b not in node_set:
        raise ValueError(f"Unknown benchmark '{b}'. Available: {nodes}")
    return b

# ----------------------------
# A) Absolute height anchors
# ----------------------------
anchors = pd.DataFrame(columns=["benchmark", "H_abs_m", "sigma_H_m", "source"])

has_anchors = _ask_yes_no("\nDo you have absolute height(s) for any benchmark(s)? (y/n): ")
if has_anchors:
    nA = _read_int("How many benchmarks have absolute heights? (integer): ", min_val=1)
    rows = []
    print(f"\nAvailable benchmarks: {nodes}")
    for k in range(nA):
        print(f"\nAnchor {k+1}/{nA}")
        b = _read_benchmark("Benchmark name: ")
        H = _read_float("Absolute height H [m]: ")
        sig_mm = _read_float("Uncertainty sigma(H) [mm]: ")
        src = input("Source/method (free text, e.g., GNSSBM ITRF->EVRF): ").strip()
        rows.append({"benchmark": b, "H_abs_m": H, "sigma_H_m": sig_mm/1000.0, "source": src})
    anchors = pd.DataFrame(rows).drop_duplicates(subset=["benchmark"], keep="last").reset_index(drop=True)

print("\nAnchors DataFrame: anchors")
display(anchors)

# ----------------------------
# B) External ΔH observations
# ----------------------------
external_dh = pd.DataFrame(columns=["from", "to", "dH_m", "sigma_dH_m", "method", "date", "notes"])

has_ext = _ask_yes_no("\nDo you have external height differences ΔH between benchmarks (other campaigns/methods)? (y/n): ")
if has_ext:
    nE = _read_int("How many external ΔH observations will you enter? (integer): ", min_val=1)
    rows = []
    print(f"\nAvailable benchmarks: {nodes}")
    for k in range(nE):
        print(f"\nExternal ΔH {k+1}/{nE}")
        f = _read_benchmark("From benchmark: ")
        t = _read_benchmark("To benchmark: ")
        dH = _read_float("ΔH = H_to - H_from [m]: ")
        sig_mm = _read_float("Uncertainty sigma(ΔH) [mm]: ")
        method = input("Method/campaign (e.g., leveling 2025-12, DGPS, TS campaign B): ").strip()
        date = input("Date (optional, YYYY-MM-DD or free text): ").strip()
        notes = input("Notes (optional): ").strip()
        rows.append({
            "from": f, "to": t, "dH_m": dH, "sigma_dH_m": sig_mm/1000.0,
            "method": method, "date": date if date else None, "notes": notes if notes else None
        })
    external_dh = pd.DataFrame(rows).reset_index(drop=True)

print("\nExternal ΔH DataFrame: external_dh")
display(external_dh)

# ----------------------------
# Save for later
# ----------------------------
constraints = {
    "anchors": anchors,
    "external_dh": external_dh
}
print("\nSaved constraints in dict: constraints")



Do you have absolute height(s) for any benchmark(s)? (y/n): y
How many benchmarks have absolute heights? (integer): 1

Available benchmarks: ['F1', 'F2', 'R1', 'R2', 'R3', 'R4', 'R5', 'T1', 'T2', 'T3']

Anchor 1/1
Benchmark name: R1
Absolute height H [m]: 43.9171
Uncertainty sigma(H) [mm]: 12
Source/method (free text, e.g., GNSSBM ITRF->EVRF): GINAN PPP

Anchors DataFrame: anchors


,benchmark,H_abs_m,sigma_H_m,source
0,R1,43.9171,0.012,GINAN PPP



Do you have external height differences ΔH between benchmarks (other campaigns/methods)? (y/n): n

External ΔH DataFrame: external_dh


,from,to,dH_m,sigma_dH_m,method,date,notes



Saved constraints in dict: constraints
